In [ ]:
# HORIZON runner (docs/AUTOMATION.md, Section 3).  It clones the public repository, runs the chunk that
# jobs/state.json dispatches, with the code at the manifest's tag, and leaves result.json,
# bundle.tar.gz and console.log in the working folder.  It holds no credential.
import json, os, shutil, subprocess, sys, time
T0 = time.time()                                     # the session deadline is computed from this
REPO_URL = os.environ.get("HORIZON_REPO_URL", "https://github.com/rahul-k-mishra/horizon.git")
WORKING = os.environ.get("HORIZON_WORKING", "/kaggle/working")
TMP = os.environ.get("HORIZON_TMP", "/tmp")
STATE_DIR, CODE_DIR = os.path.join(TMP, "horizon-state"), os.path.join(TMP, "horizon-code")
os.makedirs(WORKING, exist_ok=True)
LOG = open(os.path.join(WORKING, "console.log"), "w", encoding="utf-8")

def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    LOG.write(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    LOG.flush()
    print(r.stdout, r.stderr, flush=True)
    if r.returncode:
        raise SystemExit(f"command failed with code {r.returncode}: {' '.join(cmd)}")
    return r

for d in (STATE_DIR, CODE_DIR):
    shutil.rmtree(d, ignore_errors=True)
sh("git", "clone", "--quiet", REPO_URL, STATE_DIR)
TAG = json.load(open(os.path.join(STATE_DIR, "jobs", "manifest.json"), encoding="utf-8"))["code_tag"]
sh("git", "-C", STATE_DIR, "worktree", "add", "--quiet", "--detach", CODE_DIR, TAG)
sh("git", "-C", STATE_DIR, "log", "-1", "--format=state commit %H %s")
sh("git", "-C", CODE_DIR, "log", "-1", "--format=code commit %H (tag " + TAG + ")")
REQ = os.path.join(CODE_DIR, "kaggle", "requirements-runner.txt")
if any(l.strip() and not l.lstrip().startswith("#") for l in open(REQ, encoding="utf-8")):
    sh(sys.executable, "-m", "pip", "install", "--quiet", "-r", REQ)
sh(sys.executable, os.path.join(CODE_DIR, "scripts", "automation", "runner.py"), "--state-repo", STATE_DIR,
   "--out", WORKING, "--work-dir", os.path.join(TMP, "horizon-work"), "--t0", repr(T0))
LOG.close()
